# 🧮 Math Escalation — Full LLM Training (Standalone Colab)

This notebook trains a real LLM using **GRPO + Unsloth** on adaptive math problems.
It is **fully self-contained** — it generates problems and verifies rewards internally,
so you don't need a running server.

### 🌟 Stack:
- **Unsloth** — 2x faster training, 70% less VRAM
- **Qwen2.5-1.5B-Instruct** — fast, capable base model
- **TRL GRPO** — Group Relative Policy Optimization
- **10-Tier Curriculum** — problems escalate as the model improves
- **Multi-component Reward** — correctness + reasoning format

In [ ]:
# Cell 1 — Install Dependencies
!pip install "unsloth[colab-new]" trl>=0.12.0 datasets -q
print('✅ Dependencies installed!')

In [ ]:
# Cell 2 — Embedded Math Environment (No Server Needed)
import random
import re
import math

def generate_problem(difficulty: int):
    """Generate a math problem for the given tier (1-10) and return (problem_str, answer)."""
    d = difficulty
    if d == 1:
        a, b = random.randint(1, 10), random.randint(1, 10)
        return f"{a} + {b}", float(a + b)
    elif d == 2:
        a, b = random.randint(10, 99), random.randint(10, 99)
        return f"{a} + {b}", float(a + b)
    elif d == 3:
        a, b = sorted([random.randint(10, 99), random.randint(10, 99)], reverse=True)
        return f"{a} - {b}", float(a - b)
    elif d == 4:
        a, b = random.randint(2, 12), random.randint(2, 12)
        return f"{a} * {b}", float(a * b)
    elif d == 5:
        a, b, c = random.randint(2, 10), random.randint(2, 10), random.randint(1, 20)
        return f"{a} * {b} + {c}", float(a * b + c)
    elif d == 6:
        a, b, c = random.randint(2, 10), random.randint(2, 10), random.randint(1, 20)
        return f"{a} * ({b} + {c})", float(a * (b + c))
    elif d == 7:
        a, b = random.randint(5, 20), random.randint(2, 9)
        return f"{a * b} / {b}", float(a)
    elif d == 8:
        a, x = random.randint(2, 9), random.randint(1, 15)
        b = random.randint(1, 30)
        c = a * x + b
        return f"Solve for x: {a}x + {b} = {c}", float(x)
    elif d == 9:
        x = random.randint(2, 20)
        return f"sqrt({x * x})", float(x)
    else:
        a, x_val, div = random.randint(2, 5), random.randint(1, 10), random.randint(2, 4)
        b = div * random.randint(2, 8) - a * x_val
        e = (a * x_val + b) // div
        return f"Solve for x: ({a}x + {b}) / {div} = {e}", float(x_val)

def check_answer(problem_str: str, expected: float, model_answer: float) -> float:
    """Returns reward: +1.0 correct, -0.5 wrong."""
    return 1.0 if abs(model_answer - expected) < 1e-4 else -0.5

print('✅ Math environment ready! Tiers 1-10 embedded.')

In [ ]:
# Cell 3 — Load Model with Unsloth (4-bit quantized for T4)
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

print('Loading Qwen2.5-1.5B-Instruct with Unsloth 4-bit...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

print(f'✅ Model loaded! Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 4 — Multi-Component Reward Function
import re

def compute_reward(completions, prompts, answers, **kwargs):
    """
    Reward function called by GRPOTrainer.
    answers: list of expected float answers passed via dataset column.
    Returns rewards for each completion.
    """
    rewards = []
    for completion, prompt, expected in zip(completions, prompts, answers):
        text = completion[0]['content'] if isinstance(completion, list) else str(completion)

        # Component 1: Reasoning format reward
        format_reward = 0.0
        if '<thought>' in text and '</thought>' in text:
            format_reward += 0.5
            m = re.search(r'<thought>(.*?)</thought>', text, re.DOTALL)
            if m and len(m.group(1).strip()) > 10:
                format_reward += 0.2  # Bonus for substantive reasoning

        # Component 2: Numeric presence reward
        # Strip reasoning tags before looking for answer
        clean = re.sub(r'<thought>.*?</thought>', '', text, flags=re.DOTALL)
        nums = re.findall(r'-?\d+\.?\d*', clean)
        numeric_reward = 0.3 if nums else 0.0

        # Component 3: Correctness reward (Environment verifier)
        env_reward = -0.5
        if nums:
            try:
                predicted = float(nums[-1])
                expected_val = float(expected)
                env_reward = check_answer('', expected_val, predicted)
            except Exception:
                env_reward = -0.5

        total = env_reward + format_reward + numeric_reward
        rewards.append(total)

    return rewards

print('✅ Reward function defined (correctness + format + numeric = max 2.0)')

In [ ]:
# Cell 5 — Build Curriculum Dataset (Embedded, No Server)
import datasets

def build_curriculum_dataset(n_problems=600):
    """
    Build a training dataset by sampling problems across all 10 difficulty tiers.
    Distributes problems so harder tiers are also represented.
    """
    print(f'📊 Building curriculum dataset with {n_problems} problems...')
    data = []

    # Distribution: more easy problems early, spread across all tiers
    tier_counts = {
        1: 100, 2: 80, 3: 80, 4: 70, 5: 60,
        6: 50, 7: 50, 8: 40, 9: 40, 10: 30
    }

    for tier, count in tier_counts.items():
        for _ in range(count):
            problem_str, answer = generate_problem(tier)
            prompt = (
                f'Solve the following math problem. '
                f'Think step-by-step inside <thought> tags, '
                f'then give ONLY the final numeric answer.\n\n'
                f'[Tier {tier}/10] Problem: {problem_str}\n'
                f'Answer:'
            )
            data.append({'prompt': prompt, 'answer': str(answer)})

    # Shuffle to mix tiers
    random.shuffle(data)
    ds = datasets.Dataset.from_list(data)
    print(f'✅ Dataset ready: {len(ds)} problems across Tiers 1-10')

    # Show sample
    sample = ds[0]
    print(f"\nSample prompt: {sample['prompt'][:200]}...")
    print(f"Expected answer: {sample['answer']}")
    return ds

train_dataset = build_curriculum_dataset(600)
print(f'Dataset size: {len(train_dataset)}')

In [ ]:
# Cell 6 — GRPO Training Configuration & Launch
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir = './math_escalation_grpo',
    num_train_epochs = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    learning_rate = 5e-6,
    logging_steps = 5,
    save_steps = 100,
    warmup_ratio = 0.05,
    lr_scheduler_type = 'cosine',
    report_to = 'none',
    # GRPO Parameters
    num_generations = 8,
    max_completion_length = 256,
    temperature = 0.9,
    # Memory
    gradient_checkpointing = True,
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    reward_funcs = compute_reward,
    processing_class = tokenizer,
)

print('🔥 Starting GRPO Training...')
print('Expected: ~600 steps | ~30-60 min on T4')
trainer.train()
print('✅ Training complete!')

In [ ]:
# Cell 7 — Save Merged Model (Judge-Ready)
print('💾 Saving merged model...')
model.save_pretrained_merged('./math_grpo_merged', tokenizer, save_method='merged_16bit')
print('✅ Model saved at ./math_grpo_merged')
print('To use: AutoModelForCausalLM.from_pretrained("./math_grpo_merged")')

In [ ]:
# Cell 8 — Quick Eval: Test the Trained Model
FastLanguageModel.for_inference(model)

def test_model(problem_str, tier=1):
    prompt = (
        f'Solve the following math problem. '
        f'Think step-by-step inside <thought> tags, '
        f'then give ONLY the final numeric answer.\n\n'
        f'[Tier {tier}/10] Problem: {problem_str}\n'
        f'Answer:'
    )
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Test on all tiers
test_cases = [
    ("5 + 7", 1, 12.0),
    ("45 + 38", 2, 83.0),
    ("6 * 8", 4, 48.0),
    ("3 * (4 + 5)", 6, 27.0),
    ("sqrt(49)", 9, 7.0),
]

print('\n📊 Post-Training Evaluation:')
print('='*50)
correct = 0
for problem, tier, expected in test_cases:
    response = test_model(problem, tier)
    nums = re.findall(r'-?\d+\.?\d*', re.sub(r'<thought>.*?</thought>', '', response, flags=re.DOTALL))
    predicted = float(nums[-1]) if nums else None
    is_correct = predicted is not None and abs(predicted - expected) < 1e-4
    if is_correct:
        correct += 1
    status = '✅' if is_correct else '❌'
    print(f'{status} [{tier}] {problem} = {expected} | Model: {predicted}')
    print(f'   Response: {response[:100]}...')
    print()

print(f'Score: {correct}/{len(test_cases)} ({100*correct//len(test_cases)}%)')